# siope_entrate_comuni - notebook v0

Validazione della pipeline per fasi: **raw → clean → mart**.

- scopo: verificare che ogni layer sia corretto e coerente con il precedente
- le SQL sono la fonte di verità: i check numerici devono essere letti alla luce di quello che dichiarano
- non sostituisce l'analisi pubblica
- evitare output pesanti o immagini embeddate nel commit

In [1]:
import re
import yaml
import pandas as pd
from pathlib import Path

# --- Unici parametri da impostare manualmente ---
METRICA       = "importo_totale"  # colonna numerica nel mart (tabella agg)
METRICA_CLEAN = "importo"        # colonna nel clean

# --- Anchor: usa il path del notebook se disponibile (VSCode), altrimenti cwd ---
try:
    _start = Path(__vsc_ipynb_file__).resolve().parent  # VSCode Jupyter
except NameError:
    _start = Path.cwd().resolve()

# Walk up finché non troviamo dataset.yml
candidate_dir = None
for probe in [_start, *_start.parents]:
    if (probe / "dataset.yml").exists():
        candidate_dir = probe
        break

if candidate_dir is None:
    raise FileNotFoundError(f"dataset.yml non trovato risalendo da {_start}")

cfg = yaml.safe_load((candidate_dir / "dataset.yml").read_text())

SLUG       = cfg["dataset"]["name"]
ANNO_RUN   = cfg["dataset"]["years"][-1]
MART_TABLE = "siope_entrate_comuni_agg_labeled"  # aggiornato: _agg non piu` generato
ENCODING   = cfg.get("clean", {}).get("read", {}).get("encoding", "utf-8")
DELIM      = cfg.get("clean", {}).get("read", {}).get("delim", ",")
HEADER     = cfg.get("clean", {}).get("read", {}).get("header", True)
SKIP       = cfg.get("clean", {}).get("read", {}).get("skip", 0)

DI_ROOT   = (candidate_dir / cfg["root"]).resolve()
RAW_DIR   = DI_ROOT / "data" / "raw"   / SLUG / str(ANNO_RUN)
CLEAN_DIR = DI_ROOT / "data" / "clean" / SLUG / str(ANNO_RUN)
MART_DIR  = DI_ROOT / "data" / "mart"  / SLUG / str(ANNO_RUN)
SQL_DIR   = candidate_dir / "sql"

print(f"slug      : {SLUG}")
print(f"anno_run  : {ANNO_RUN}")
print(f"mart table: {MART_TABLE}")
print(f"encoding  : {ENCODING}  delim: {repr(DELIM)}")
print(f"header    : {HEADER}  skip: {SKIP}")

slug      : siope_entrate_comuni
anno_run  : 2025
mart table: siope_entrate_comuni_agg
encoding  : utf-8  delim: ','
header    : False  skip: 0


## SQL di riferimento

Le SQL sono la fonte di verità per capire cosa deve contenere ogni layer.
Leggile prima di interpretare i check numerici.

In [2]:
for sql_file in sorted(SQL_DIR.glob("*.sql")):
    print(f"{'='*60}")
    print(f"  {sql_file.name}")
    print(f"{'='*60}")
    print(sql_file.read_text())
    print()

  clean.sql
select
    trim(column0) as codice_ente,
    try_cast(column1 as integer) as anno,
    trim(column2) as periodo,
    trim(column3) as codice_voce,
    try_cast(column4 as bigint) as importo
from raw_input;


  mart_comuni.sql
with anag_enti_seed as (
    select
        *,
        regexp_extract(replace(filename, '\', '/'), '.*/([0-9]{4})/[^/]+$', 1)::integer as snapshot_year
    from read_parquet('{support.enti.mart}', filename = true)
),
anag_enti as (
    select * exclude (filename, snapshot_year)
    from anag_enti_seed
    where snapshot_year = (select max(snapshot_year) from anag_enti_seed)
),
sottocomparti_seed as (
    select
        *,
        regexp_extract(replace(filename, '\', '/'), '.*/([0-9]{4})/[^/]+$', 1)::integer as snapshot_year
    from read_parquet('{support.sottocomparti.mart}', filename = true)
),
sottocomparti_map as (
    select * exclude (filename, snapshot_year)
    from sottocomparti_seed
    where snapshot_year = (select max(snapshot_year) from s

## 1. Raw

Verifica che il file raw esista, sia leggibile con i parametri dichiarati in `dataset.yml` e abbia il numero di righe atteso.

In [3]:
raw_files = sorted(RAW_DIR.glob("*.csv")) + sorted(RAW_DIR.glob("*.xlsx")) + sorted(RAW_DIR.glob("*.parquet"))
if not raw_files:
    raise FileNotFoundError(f"Nessun file raw trovato in {RAW_DIR}")

raw_path = raw_files[0]
print(f"File: {raw_path.name}  ({raw_path.stat().st_size / 1024:.0f} KB)")

N_RAW = None
raw_full = None
try:
    if raw_path.suffix == ".parquet":
        raw_full = pd.read_parquet(raw_path)
    elif raw_path.suffix in (".csv", ".tsv"):
        header_row = 0 if HEADER else None
        raw_full = pd.read_csv(raw_path, encoding=ENCODING, sep=DELIM, header=header_row, skiprows=SKIP)
    elif raw_path.suffix == ".xlsx":
        raw_full = pd.read_excel(raw_path, header=0 if HEADER else None, skiprows=SKIP)
    N_RAW = len(raw_full)
    print(f"Righe raw   : {N_RAW}")
    print(f"Colonne raw : {len(raw_full.columns)} -> {list(raw_full.columns)}")
    print("Raw caricato OK")
    raw_full.head(3)
except Exception as e:
    print(f"WARNING: raw non leggibile con pandas ({e})")
    print("-> Skip raw-load, il clean parquet e gia validato dalla pipeline.")
    N_RAW = None

File: ENTRATE_2025.D260430.H2042.csv  (201813 KB)
Righe raw   : 4286586
Colonne raw : 5 -> [0, 1, 2, 3, 4]
Raw caricato OK


## 2. Clean

Verifica schema e null. I null su colonne `try_cast` sono attesi se il raw contiene valori non parsabili.
Il confronto righe raw vs clean mostra quante righe sono state filtrate dal `WHERE` della clean.sql.

In [4]:
clean_files = sorted(CLEAN_DIR.glob("*.parquet"))
if not clean_files:
    raise FileNotFoundError(f"Clean non trovato in {CLEAN_DIR}. Eseguire: toolkit run clean")

clean = pd.read_parquet(clean_files[0])
N_CLEAN = len(clean)

print(f"Righe clean : {N_CLEAN}")
print(f"Colonne     : {list(clean.columns)}")
clean.head(3)

Righe clean : 4286586
Colonne     : ['codice_ente', 'anno', 'periodo', 'codice_voce', 'importo']


,codice_ente,anno,periodo,codice_voce,importo
0,000008927000098,2025,07,9.01.01.02.001,93185
1,800000619,2025,03,2.01.01.02.003,26376682
2,022340935000471,2025,12,9999,134522797


In [5]:
if N_RAW is not None:
    dropped = N_RAW - N_CLEAN
    dropped_pct = dropped / N_RAW * 100 if N_RAW > 0 else 0
    print(f"Righe raw         : {N_RAW}")
    print(f"Righe clean       : {N_CLEAN}")
    print(f"Righe filtrate    : {dropped} ({dropped_pct:.1f}%)")
    print()
    if dropped > 0:
        print("-> Verificare la WHERE in clean.sql per capire quali righe vengono escluse.")
    else:
        print("-> Nessuna riga filtrata: clean e fedele al raw.")
else:
    print(f"Raw non caricato (parsing error) -- righe clean: {N_CLEAN}")
    print("-> Check raw-vs-clean saltato. Clean gia validato dalla pipeline.")

print("\nNull per colonna clean:")
null_counts = clean.isnull().sum()
print(null_counts[null_counts > 0].to_string() if null_counts.any() else "  nessuno")

Righe raw         : 4286586
Righe clean       : 4286586
Righe filtrate    : 0 (0.0%)

-> Nessuna riga filtrata: clean e fedele al raw.

Null per colonna clean:
  nessuno


## 3. Mart

Verifica unicità sulle chiavi del GROUP BY, anni presenti, null e consistenza dei totali con il clean.

In [6]:
mart_path = MART_DIR / f"{MART_TABLE}.parquet"
if not mart_path.exists():
    raise FileNotFoundError(f"Mart non trovato: {mart_path}. Eseguire: toolkit run mart")

mart = pd.read_parquet(mart_path)
print(f"Righe mart  : {len(mart)}")
print(f"Colonne     : {list(mart.columns)}")
mart.head(3)

Righe mart  : 373165
Colonne     : ['codice_ente', 'anno', 'codice_voce', 'denominazione_ente', 'tipo_ente', 'codice_sottocomparto', 'descrizione_sottocomparto', 'codice_comparto', 'descrizione_comparto', 'importo_totale', 'righe', 'periodi_coperti', 'periodo_min', 'periodo_max']


,codice_ente,anno,codice_voce,denominazione_ente,tipo_ente,codice_sottocomparto,descrizione_sottocomparto,codice_comparto,descrizione_comparto,importo_totale,righe,periodi_coperti,periodo_min,periodo_max
0,011116655,2025,4.02.01.01.001,COMUNE DI MOLVENO,COMUNE,COMUNE,COMUNI,PRO,Province - Comuni - Citta' metropolitane - Uni...,116549988.0,12,12,01,12
1,011116652,2025,3.01.03.01.001,COMUNE DI BARETE,COMUNE,COMUNE,COMUNI,PRO,Province - Comuni - Citta' metropolitane - Uni...,1333100.0,11,11,02,12
2,011120882,2025,3.01.02.01.033,COMUNE DI MONASTEROLO DI SAVIGLIANO,COMUNE,COMUNE,COMUNI,PRO,Province - Comuni - Citta' metropolitane - Uni...,92700.0,12,12,01,12


In [7]:
# Estrai chiavi GROUP BY dalla mart.sql per il check di unicità.
# Per query con CTE, cerca GROUP BY solo nella SELECT finale (dopo tutti i CTE).
# Se la SELECT finale non ha GROUP BY, il check di unicità va fatto manualmente.
mart_sql_path = SQL_DIR / Path(cfg["mart"]["tables"][0]["sql"]).name
groupby_keys = []
if mart_sql_path.exists():
    sql_text = mart_sql_path.read_text()

    # Individua dove inizia la SELECT finale (depth 0, dopo eventuali CTE)
    sql_body = sql_text
    if re.search(r'\bwith\b', sql_body, re.IGNORECASE):
        depth = 0
        final_select_pos = None
        for tok in re.finditer(r'[()]|\bselect\b', sql_body, re.IGNORECASE):
            ch = tok.group()
            if ch == '(':
                depth += 1
            elif ch == ')':
                depth -= 1
            elif ch.lower() == 'select' and depth == 0:
                final_select_pos = tok.start()
        if final_select_pos is not None:
            sql_body = sql_body[final_select_pos:]

    match = re.search(r'group\s+by\s+([\d\s,]+)', sql_body, re.IGNORECASE)
    if match:
        positions = [int(p.strip()) for p in match.group(1).split(',')]
        groupby_keys = [mart.columns[i - 1] for i in positions if i <= len(mart.columns)]
    else:
        match = re.search(r'group\s+by\s+((?:[\w.]+(?:\s*,\s*)?)+)', sql_body, re.IGNORECASE)
        if match:
            groupby_keys = [k.strip().split('.')[-1] for k in match.group(1).split(',')]
            groupby_keys = [k for k in groupby_keys if k in mart.columns]

if groupby_keys:
    dups = mart.duplicated(subset=groupby_keys).sum()
    print(f"Chiavi GROUP BY (SELECT finale): {groupby_keys}")
    print(f"Duplicati                       : {dups}")
    assert dups == 0, f"FAIL: {dups} righe duplicate sulle chiavi del mart -- verificare mart.sql"
    print("OK: nessun duplicato sulle chiavi.")
else:
    print("GROUP BY non trovato nella SELECT finale -- check duplicati saltato.")

GROUP BY non trovato nella SELECT finale -- check duplicati saltato.


In [8]:
if "anno" in mart.columns:
    anni_mart = sorted(mart["anno"].unique())
    print(f"Anni nel mart ({len(anni_mart)}): {anni_mart}")

null_mart = mart.isnull().sum()
print("\nNull per colonna mart:")
print(null_mart[null_mart > 0].to_string() if null_mart.any() else "  nessuno")

Anni nel mart (1): [np.int32(2025)]

Null per colonna mart:
  nessuno


In [9]:
if METRICA in mart.columns and METRICA_CLEAN in clean.columns:
    tot_mart  = mart[METRICA].sum()
    tot_clean = clean[METRICA_CLEAN].sum()
    # NOTE: il mart filtra per COMUNE + codice_comparto=PRO, clean no.
    # Il delta e atteso: i totali non coincidono perche il mart e un
    # sottoinsieme filtrato. Non usare questo check per validare la pipeline.
    # Verificare piuttosto il numero di comuni unici e la copertura periodi.
    print(f"Totale mart  ({METRICA})      : {tot_mart:,.2f}")
    print(f"Totale clean ({METRICA_CLEAN}): {tot_clean:,.2f}")
    print(f"Nota: il mart e filtrato (COMUNE/PRO) -- totali non confrontabili")

else:
    print("Nota: il mart filtra per COMUNE/PRO -- totali non confrontabili.")
    print("Check non applicabile a questo dataset.")

Totale mart  (importo_totale)      : 11,592,627,575,949.00
Totale clean (importo): 63,469,132,481,999.00
Nota: il mart e filtrato (COMUNE/PRO) -- totali non confrontabili


In [10]:
PERIMETRO = "comuni, entrate, 2021-2025, SIOPE aperto"

print(f"Slug            : {SLUG}")
print(f"Anno run        : {ANNO_RUN}")
print(f"Tabella mart    : {MART_TABLE}")
print(f"Metrica mart    : {METRICA}")
print(f"Metrica clean   : {METRICA_CLEAN}")
print(f"Perimetro       : {PERIMETRO}")

Slug            : siope_entrate_comuni
Anno run        : 2025
Tabella mart    : siope_entrate_comuni_agg
Metrica mart    : importo_totale
Metrica clean   : importo
Perimetro       : comuni, entrate, 2021-2025, SIOPE aperto
